# Trabajo del grupo 6, datos de las 5 grandes ligas europeas del fútbol
Alejandro Álvarez Segoviano e Ivo Reguero Ferrer

# 0. Importamos librerías

In [13]:
import pandas as pd
from pathlib import Path

In [14]:
ruta = Path().resolve().parent
ruta = ruta / "data" / "fulbo.csv"
ruta

PosixPath('/home/alumnos/ivoreg/PycharmProjects/Proyecto-Fusball/data/fulbo.csv')

# 1. Primer vistazo a los datos y EDA

## 1.1 Mirada inicial

Vamos a comenzar leyendo los datos, observando de cuántas observaciones disponemos, cuántos atributos y de qué naturaleza, previo a la modelización para entender mejor la naturaleza de ellas, cómo se relacionan y si presentan limitaciones o sesgos, para hallar de que tipo son.

In [15]:
df = pd.read_csv(ruta,header=1)

In [16]:
df.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,1,Brenden Aaronson,us USA,MFFW,Leeds United,eng Premier League,24.0,2000.0,37,30,...,0,3,0,0.15,0.18,0.33,0.15,0.33,Matches,5bc43860
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,4f504fcc
2,3,Zach Abbott,eng ENG,DF,Nottingham,eng Premier League,19.0,2006.0,3,2,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,c103921a
3,4,Jones El-Abdellaoui,ma MAR,MFFW,Celta Vigo,es La Liga,19.0,2006.0,22,4,...,0,0,0,0.26,0.00,0.26,0.26,0.26,Matches,29dd3b2e
4,5,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,...,0,1,0,0.00,0.00,0.00,0.00,0.00,Matches,7978cbf0


In [17]:
print(f"Disponemos de {df.shape[0]} jugadores y {df.shape[1]} atributos.")

Disponemos de 2839 jugadores y 27 atributos.


Esos 27 atributos son los siguientes :

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2839 entries, 0 to 2838
Data columns (total 27 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Rk       2839 non-null   int64  
 1   Player   2839 non-null   str    
 2   Nation   2836 non-null   str    
 3   Pos      2839 non-null   str    
 4   Squad    2839 non-null   str    
 5   Comp     2839 non-null   str    
 6   Age      2837 non-null   float64
 7   Born     2837 non-null   float64
 8   MP       2839 non-null   int64  
 9   Starts   2839 non-null   int64  
 10  Min      2839 non-null   int64  
 11  90s      2839 non-null   float64
 12  Gls      2839 non-null   int64  
 13  Ast      2839 non-null   int64  
 14  G+A      2839 non-null   int64  
 15  G-PK     2839 non-null   int64  
 16  PK       2839 non-null   int64  
 17  PKatt    2839 non-null   int64  
 18  CrdY     2839 non-null   int64  
 19  CrdR     2839 non-null   int64  
 20  Gls.1    2839 non-null   float64
 21  Ast.1    2839 non-null   

Algunos atributos tienen nulos, de igual forma cada variable individualmente será revisada después por si hay nulos codificados.

## 1.2 Borrando columnas que no nos dan información

La página web de la que provienen los datos posee un glosario que usaremos para poder explicar qué significa cada dato, en la siguiente sección lo usaremos.

Tras haber sido transformado a un CSV, hemos perdido información a la que podríamos acceder desde la página web: enlaces en el nombre de los jugadores que nos permitían acceder a un perfil de la página del jugador con más información, lo cual ocurre también con la selección, el equipo y la liga, y la razón por la que se ha escogido esa nacionalidad para el jugador.

Otro dato que perdemos es el enlace que te conecta a los partidos que ha jugado en esa temporada el jugador. Si quisiésemos recuperar el enlace, en el caso de Brenden Aaronson, tendríamos que reconstruirlo usando el atributo de la última columna, la temporada que queremos mirar (en nuestro caso 2025-2026) y su nombre separado por guiones para poder ver todos los partidos que jugó durante la temporada. (https://fbref.com/en/players/5bc43860/matchlogs/2025-2026/summary/Brenden-Aaronson-Match-Logs)

Ahora que sabemos que estas columnas no nos sirven para modelar, podemos borrar las columnas _Matches_ y _-9999_.

Un dato más que no nos sirve es _Rk_, un dato usado dentro de la página web para ver el la posición del jugador cuando estás ordenando dependiendo de un dato (si ordenas por más partidos jugados, _Rk_ se actualiza en la página web). Es decir, es un índice, es rendundante y no lo queremos.

**Vamos a borrar esas tres columnas:**

In [19]:
df = df.drop(columns=['Rk', 'Matches', '-9999'])
df.head()

,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,...,G-PK,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,Brenden Aaronson,us USA,MFFW,Leeds United,eng Premier League,24.0,2000.0,37,30,2449,...,4,0,0,3,0,0.15,0.18,0.33,0.15,0.33
1,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,17,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00
2,Zach Abbott,eng ENG,DF,Nottingham,eng Premier League,19.0,2006.0,3,2,164,...,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00
3,Jones El-Abdellaoui,ma MAR,MFFW,Celta Vigo,es La Liga,19.0,2006.0,22,4,681,...,2,0,0,0,0,0.26,0.00,0.26,0.26,0.26
4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,138,...,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00


Tal y como se indica en la página web, y como se muestra en el csv antes de ser procesado, hay cuatro grupos de columnas:
- (sin nombre oficial) Datos de "fuera del campo" del jugador (a la que pertenecen _Player_, _Nation_, _Pos_, _Squad_, _Comp_, _Age_ y _Born_)
- _Playing Time_ (tiempo de juego, pertenecen _MP_, _Starts_, _Min_ y _90s_)
- _Performance_ (rendimiento, pertenecen _Gls_, _Ast_, _G+A_, _G-PK_, _PK_, _PKatt_, _CrdY_ y _CrdR_)
- _Per 90 Minutes_ (por 90 minutos, pertenecen _Gls.1_, _Ast.1_, _G+A.1_, _G-PK.1_ y _G+A-PK_)

Sin siquiera haber empezado a explicar qué significa cada atributo, vemos que muchos son fabricados a partir de otros (muchos son simples divisiones de algún atributo entre _Min_).

## 1.3 Explicación de cada atributo

_Player_: El nombre del jugador. Puede haber repetidos, ya que un mismo jugador puede haber jugado en la misma temporada en dos clubs distintos (hay dos Himad Abdelli con la misma edad y club distinto, lo más probable es que sea la misma persona que ha cambiado de club durante el mercado de fichajes de invierno), y dos jugadores distintos pueden tener el mismo nombre (hay dos David López con una edad muy distinta)

_Nation_: Nacionalidad del jugador. La nacionalidad es escogida comprobando para cuál selección han jugado, y si no han jugado para ninguna se escoge dependiendo de la nacionalidad indicada en Wikipedia o su lugar de nacimiento si está disponible.

_Pos_: la posición en la que juega el jugador. Puede ser una o más de estas, las cuales no están separadas por espacios:

- GK - Portero
- DF - Defensa
- MF - Mediocentro
- FW - Delantero

Hemos borrado de esta lista, sacada del glosario, los roles que no aparecen representados en la base de datos. En este código se puede ver que solamente aparecen esas 4 posiciones y algunas combinaciones:

In [23]:
df["Pos"].value_counts()

Pos
MF      923
DF      705
FW      395
MFFW    217
GK      194
FWMF    146
DFMF    141
MFDF    115
DFFW      3
Name: count, dtype: int64

_Squad_: El equipo para el que han jugado. Puede no ser toda la temporada, como se menciona en la sección de _Player_.

_Comp_: la competición a la que pertenece el equipo. Son 5, como era de esperar:

In [24]:
df["Comp"].value_counts()

Comp
es La Liga            611
it Serie A            609
fr Ligue 1            561
eng Premier League    551
de Bundesliga         507
Name: count, dtype: int64

_Age_: Edad del jugador.

_Born_: Año de nacimiento del jugador. Vamos a ver rápidamente si los dos atributos son calculados el uno con el otro:

In [27]:
df["Age"].value_counts()

Age
22.0    244
25.0    231
24.0    224
27.0    218
23.0    211
28.0    192
26.0    189
20.0    186
21.0    176
19.0    161
18.0    117
29.0    115
30.0    111
31.0    109
32.0     93
17.0     66
33.0     57
34.0     44
35.0     29
16.0     19
36.0     13
38.0     10
37.0      8
15.0      6
39.0      6
40.0      1
41.0      1
Name: count, dtype: int64

In [28]:
df["Born"].value_counts()

Born
2002.0    234
2003.0    230
2000.0    227
2001.0    214
1999.0    207
1997.0    204
1998.0    196
2004.0    169
2005.0    165
2006.0    158
1996.0    148
1994.0    115
2007.0    110
1995.0    108
1993.0     95
1992.0     73
1991.0     46
2008.0     42
1990.0     33
1989.0     15
2009.0     14
1988.0     14
1987.0      9
1986.0      5
1985.0      3
1984.0      1
1983.0      1
2010.0      1
Name: count, dtype: int64

Al haber 6 jugadores que tienen 15 años y un solo jugador nacido en 2010, parece que la edad no es calculada con 2025 - _Born_. Eso sí, puede que tengamos que borrar uno de los dos en un futuro.

_MP_: Partidos jugados. Entre 1 y 38, en el caso de la liga alemana y la francesa el máximo es menos, hay menos equipos en esas ligas y menos partidos (máximo de 34).

_Starts_: Partidos en los que el jugador fue titular. Mismo rango que el atributo anterior.

_Min_: Minutos jugados a lo largo de la temporada.

_90s_ : Minutos jugados dividido por 90. Parece que no se cuentan los minutos jugados en el tiempo añadido, ya que esta métrica tiene como máximo 38.0.

-------------------------------------------------------------------------------------------------------------------------------

to do, tareas

doble analisis exploratorio para buscar un objetivo?? scatterplot?